# Warfarin QML — E02: Strong Classical Reference (Full Development)

**Phạm vi:** Thực nghiệm **E02 — Strong_Classical** trong `docs/experiment_matrix.csv`: huấn luyện **Logistic Regression, SVM-RBF, MLP** trên **4.328 bản ghi Development**, dùng **đủ 10 raw features được cho phép** sau preprocessing/one-hot/scaling, **không PCA** và **không scale góc [0,π]**. Sử dụng `development_folds.csv` (5 folds) đã khóa ở GĐ4.

**Không phải matched benchmark:** E01 sử dụng 600 mẫu và PCA k=4; E02 dùng nhiều dữ liệu hơn và không nén xuống 4 chiều. **Báo cáo E02 riêng**, không coi sự chênh lệch là so sánh tác động riêng của classical vs quantum.

**Chống leakage:** xác minh SHA-256 của raw và split; fit preprocessing **mới** trên training fold; transform validation; `final_test` chỉ được tham chiếu ở bước xác minh ID để chắc chắn không có mẫu test tham gia train/eval. Không huấn luyện, đánh giá, lựa chọn mô hình dựa trên Final Test.

**Giữ cấu hình reference:** LR (`max_iter=2000`), SVM-RBF (`C=1`, `gamma='scale'`), MLP (`hidden_layer_sizes=(32,16)`, `max_iter=500`, `early_stopping=True`, `n_iter_no_change=20`, seed 42) — tương ứng các baseline không cân bằng trọng số trong E01. Đây là mô hình đối chứng, không có tuning mới.

**Quy trình:** chạy các cell từ trên xuống; **dừng sau Cell 4 (Fold 0)** để kiểm tra, sau đó mới chạy Cell 5 (Folds 1–4) và Cell 6 (tổng hợp & lưu). Không tạo file `scripts/`. Kết quả được xuất tại `docs/results/strong_classical_e02_*` mà **không ghi đè** file nào trước đó.

**Giới hạn:** Feature Policy ở trạng thái nghiên cứu hồi cứu, temporal evidence chưa được xác minh; không dùng dự đoán này để hỗ trợ điều trị.

## Cell 1 — Môi trường, vị trí project và bảo vệ output

Dùng kernel Python của Conda `nt409` trong VS Code. Kiểm tra file kết quả chưa tồn tại trước khi chạy.

In [1]:
from pathlib import Path
from importlib.metadata import version
from time import perf_counter
import sys
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    recall_score, precision_score, confusion_matrix
)


def find_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if ((candidate / "configs/feature_policy.yaml").is_file()
                and (candidate / "data/splits/development_folds.csv").is_file()):
            return candidate
    raise FileNotFoundError("Please open this notebook in the Warfarin project workspace")

ROOT = find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.prepare_stage4 import load_and_validate_dataset, verify_raw_file
from src.preprocessing import build_preprocessor

SPLIT_DIR = ROOT / "data/splits"
OUT = ROOT / "docs/results"
CLASSES = ["Low", "Intermediate", "High"]
PREFIX = "strong_classical_e02"
OUTPUTS = {
    kind: OUT / f"{PREFIX}_{kind}.{suffix}"
    for kind, suffix in (
        ("metrics", "csv"),
        ("predictions", "csv"),
        ("confusion", "csv"),
        ("summary", "csv"),
        ("config", "json"),
    )
}
collisions = [str(path) for path in OUTPUTS.values() if path.exists()]
if collisions:
    raise FileExistsError(f"E02 output already exists; will not overwrite: {collisions}")

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("[PASS] Python:", sys.executable)
print("[PASS] Project:", ROOT)
print("[PASS] No E02 output collisions")
print("[INFO] Expected prefix:", PREFIX)

[PASS] Python: c:\Users\admin\anaconda3\envs\nt409\python.exe
[PASS] Project: D:\Warfarin-Quantum-Machine-Learning-Classification
[PASS] No E02 output collisions
[INFO] Expected prefix: strong_classical_e02


## Cell 2 — Kiểm tra source, policy, split GĐ4 và E02 protocol

**Chỉ xác minh Final Test tách biệt**; tuyệt đối không sử dụng các mẫu Test để training/validation.

In [2]:
verify_raw_file()

with (ROOT / "configs/feature_policy.yaml").open(encoding="utf-8") as f:
    policy = yaml.safe_load(f)
assert policy["training_allowed"] is True
approved = list(policy["approved_X_columns"])
assert len(approved) == 10 and len(set(approved)) == 10

matrix = pd.read_csv(ROOT / "docs/experiment_matrix.csv")
e02 = matrix.loc[matrix["experiment_id"] == "E02"]
assert len(e02) == 1
assert e02.iloc[0]["training_pool"] == "full_development"
assert e02.iloc[0]["representation"] == "Full_Features"
assert e02.iloc[0]["evaluation_method"] == "Stratified_5Fold"
assert set(e02.iloc[0]["models"].split("|")) == {"LR", "SVM", "MLP"}

manifest_path = SPLIT_DIR / "split_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["status"] == "LOCKED_BEFORE_MODEL_PERFORMANCE"
assert manifest["approved_features"] == approved
assert manifest["n_splits"] == 5
for filename, expected_hash in manifest["files"].items():
    assert sha256_file(SPLIT_DIR / filename) == expected_hash, filename

X, y = load_and_validate_dataset(approved)
labels = pd.read_csv(ROOT / "data/interim/labels.csv")
partitions = pd.read_csv(SPLIT_DIR / "final_split.csv")
dev_folds = pd.read_csv(SPLIT_DIR / "development_folds.csv")
qdev_folds = pd.read_csv(SPLIT_DIR / "quantum_dev_folds.csv")
y_array = np.asarray(y, dtype=str)

assert len(X) == len(y_array) == len(labels) == 5410
assert labels["raw_row_index"].is_unique
assert np.array_equal(y_array, labels["target_group"].astype(str).to_numpy())
assert partitions["raw_row_index"].is_unique
assert dev_folds["raw_row_index"].is_unique
assert qdev_folds["raw_row_index"].is_unique

all_ids = set(labels["raw_row_index"].astype(int))
dev_ids = set(partitions.loc[partitions["partition"] == "development", "raw_row_index"].astype(int))
test_ids = set(partitions.loc[partitions["partition"] == "final_test", "raw_row_index"].astype(int))
fold_ids = set(dev_folds["raw_row_index"].astype(int))
qdev_ids = set(qdev_folds["raw_row_index"].astype(int))
assert len(dev_ids) == 4328 and len(test_ids) == 1082
assert dev_ids.isdisjoint(test_ids) and dev_ids | test_ids == all_ids
assert fold_ids == dev_ids and len(fold_ids) == 4328
assert qdev_ids.issubset(dev_ids) and len(qdev_ids) == 600
assert set(dev_folds["fold_id"].astype(int)) == set(range(5))
assert set(dev_folds.groupby("fold_id").size()) == {865, 866}

print("[PASS] Raw/source and all split hashes")
print("[PASS] E02 protocol: full Development, full approved features, 5 folds")
print("[PASS] 4328 Development IDs, 1082 isolated Final Test IDs")
print("[PASS] Locked development fold IDs 0..4, no re-splitting")
print("[PASS] Approved 10 features; no PCA")
print("Development validation sizes:", dev_folds.groupby("fold_id").size().to_dict())

[PASS] Raw/source and all split hashes
[PASS] E02 protocol: full Development, full approved features, 5 folds
[PASS] 4328 Development IDs, 1082 isolated Final Test IDs
[PASS] Locked development fold IDs 0..4, no re-splitting
[PASS] Approved 10 features; no PCA
Development validation sizes: {0: 866, 1: 866, 2: 866, 3: 865, 4: 865}


## Cell 3 — Ba baseline và hàm chạy một fold

`build_preprocessor(approved)` được **tạo mới và fit trên training fold**. Sau encoding/standardization, dữ liệu được đưa thẳng vào mô hình; không dùng `PCA` hoặc `MinMaxScaler`. Cấu hình giống các baseline LR/SVM/MLP không weighted ở E01.

In [3]:
MODELS = ("LR", "SVM", "MLP")

def make_model(name):
    if name == "LR":
        return LogisticRegression(max_iter=2000, random_state=42)
    if name == "SVM":
        return SVC(kernel="rbf", C=1.0, gamma="scale")
    if name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(32, 16), max_iter=500,
            early_stopping=True, n_iter_no_change=20,
            random_state=42
        )
    raise ValueError(name)

# Fold outputs stay in notebook memory until final save.
fold_outputs = {}

def run_fold(fold_id):
    if fold_id in fold_outputs:
        print(f"[SKIP] Fold {fold_id} is already in notebook memory")
        return
    val_ids = set(dev_folds.loc[
        dev_folds["fold_id"] == fold_id, "raw_row_index"
    ].astype(int))
    train_ids = dev_ids - val_ids
    assert len(train_ids) + len(val_ids) == 4328
    assert len(train_ids) in (3462, 3463)
    assert len(val_ids) in (865, 866)
    assert train_ids.isdisjoint(val_ids)
    assert (train_ids | val_ids) == dev_ids
    assert (train_ids | val_ids).isdisjoint(test_ids)

    train_pos = np.flatnonzero(labels["raw_row_index"].isin(train_ids).to_numpy())
    val_pos = np.flatnonzero(labels["raw_row_index"].isin(val_ids).to_numpy())
    y_train, y_val = y_array[train_pos], y_array[val_pos]
    assert set(y_train) == set(CLASSES)
    assert set(y_val) == set(CLASSES)

    preprocess = build_preprocessor(approved)
    Z_train = np.asarray(preprocess.fit_transform(X.iloc[train_pos]), dtype=float)
    Z_val = np.asarray(preprocess.transform(X.iloc[val_pos]), dtype=float)
    assert Z_train.shape[0] == len(train_pos)
    assert Z_val.shape == (len(val_pos), Z_train.shape[1])
    assert Z_train.shape[1] > 10  # one-hot encoded, not reduced to k=4
    assert np.isfinite(Z_train).all() and np.isfinite(Z_val).all()

    fold_metrics, fold_predictions, fold_confusion = [], [], []
    for name in MODELS:
        model = make_model(name)
        start = perf_counter()
        model.fit(Z_train, y_train)
        fit_seconds = perf_counter() - start
        start = perf_counter()
        predicted = np.asarray(model.predict(Z_val), dtype=str)
        predict_seconds = perf_counter() - start
        assert len(predicted) == len(y_val)
        assert set(predicted).issubset(set(CLASSES))

        recall = recall_score(y_val, predicted, labels=CLASSES, average=None, zero_division=0)
        precision = precision_score(y_val, predicted, labels=CLASSES, average=None, zero_division=0)
        cm = confusion_matrix(y_val, predicted, labels=CLASSES)
        row = dict(
            experiment="E02", fold=fold_id, model=name,
            train_n=len(train_pos), val_n=len(val_pos),
            encoded_dim=int(Z_train.shape[1]),
            macro_f1=float(f1_score(y_val, predicted, labels=CLASSES,
                                    average="macro", zero_division=0)),
            accuracy=float(accuracy_score(y_val, predicted)),
            balanced_accuracy=float(balanced_accuracy_score(y_val, predicted)),
            recall_low=float(recall[0]), recall_intermediate=float(recall[1]),
            recall_high=float(recall[2]), precision_high=float(precision[2]),
            actual_high=int((y_val == "High").sum()),
            pred_high=int((predicted == "High").sum()),
            fit_seconds=fit_seconds, predict_seconds=predict_seconds
        )
        fold_metrics.append(row)
        for pos, actual, pred in zip(val_pos, y_val, predicted):
            fold_predictions.append(dict(
                experiment="E02", fold=fold_id, model=name,
                raw_row_index=int(labels.iloc[pos]["raw_row_index"]),
                actual=str(actual), predicted=str(pred)
            ))
        for i, actual_cls in enumerate(CLASSES):
            for j, pred_cls in enumerate(CLASSES):
                fold_confusion.append(dict(
                    experiment="E02", scope="fold", fold=fold_id,
                    model=name, actual=actual_cls,
                    predicted=pred_cls, count=int(cm[i,j])
                ))
        print(f"[PASS] E02 Fold {fold_id} | {name:<3} | "
              f"Macro-F1={row['macro_f1']:.4f} | "
              f"Accuracy={row['accuracy']:.4f} | "
              f"RecallHigh={row['recall_high']:.4f} | "
              f"fit={fit_seconds:.3f}s", flush=True)

    fold_outputs[fold_id] = (fold_metrics, fold_predictions, fold_confusion)
    print(f"[PASS] Fold {fold_id}: encoded dimension={Z_train.shape[1]}; "
          f"train={len(train_pos)}; validation={len(val_pos)}")

print("[PASS] Three E02 models configured; no PCA, no class weighting")

[PASS] Three E02 models configured; no PCA, no class weighting


## Cell 4 — Preflight: chạy **Fold 0** trước

Chỉ chạy 3 mô hình trên Fold 0 và **chưa lưu file**. Nếu PASS, gửi kết quả Fold 0 để mình kiểm tra trước khi chạy tiếp 4 folds còn lại.

In [4]:
run_fold(0)
assert 0 in fold_outputs and len(fold_outputs[0][0]) == 3
print("\n=== E02 FOLD 0 CHECKPOINT ===")
display(pd.DataFrame(fold_outputs[0][0])[
    ["fold", "model", "train_n", "val_n", "encoded_dim",
     "macro_f1", "accuracy", "recall_high", "fit_seconds"]
].round(4))
print("[PASS] E02 Fold 0 preflight completed — STOP HERE FOR REVIEW")

[PASS] E02 Fold 0 | LR  | Macro-F1=0.5851 | Accuracy=0.6721 | RecallHigh=0.2636 | fit=0.021s
[PASS] E02 Fold 0 | SVM | Macro-F1=0.5046 | Accuracy=0.6501 | RecallHigh=0.1000 | fit=0.262s
[PASS] E02 Fold 0 | MLP | Macro-F1=0.5715 | Accuracy=0.6663 | RecallHigh=0.2273 | fit=0.427s
[PASS] Fold 0: encoded dimension=42; train=3462; validation=866

=== E02 FOLD 0 CHECKPOINT ===


,fold,model,train_n,val_n,encoded_dim,macro_f1,accuracy,recall_high,fit_seconds
0,0,LR,3462,866,42,0.5851,0.6721,0.2636,0.0211
1,0,SVM,3462,866,42,0.5046,0.6501,0.1000,0.2616
2,0,MLP,3462,866,42,0.5715,0.6663,0.2273,0.4273


[PASS] E02 Fold 0 preflight completed — STOP HERE FOR REVIEW


## Cell 5 — Chỉ sau khi Fold 0 PASS: chạy Folds 1–4

Không train lại Fold 0 trong cùng phiên kernel. Chưa ghi file ra ổ đĩa; kết quả lưu trong biến `fold_outputs`.

In [5]:
assert 0 in fold_outputs, "Run Fold 0 preflight first"
for fold_id in range(1, 5):
    run_fold(fold_id)
assert set(fold_outputs) == set(range(5))
print("[PASS] E02: all 5 locked Development folds evaluated")

[PASS] E02 Fold 1 | LR  | Macro-F1=0.5772 | Accuracy=0.6755 | RecallHigh=0.2455 | fit=0.018s
[PASS] E02 Fold 1 | SVM | Macro-F1=0.5321 | Accuracy=0.6686 | RecallHigh=0.1364 | fit=0.244s
[PASS] E02 Fold 1 | MLP | Macro-F1=0.5567 | Accuracy=0.6709 | RecallHigh=0.1818 | fit=0.229s
[PASS] Fold 1: encoded dimension=42; train=3462; validation=866
[PASS] E02 Fold 2 | LR  | Macro-F1=0.5801 | Accuracy=0.6836 | RecallHigh=0.2364 | fit=0.019s
[PASS] E02 Fold 2 | SVM | Macro-F1=0.5405 | Accuracy=0.6778 | RecallHigh=0.1364 | fit=0.267s
[PASS] E02 Fold 2 | MLP | Macro-F1=0.5382 | Accuracy=0.6697 | RecallHigh=0.1545 | fit=0.185s
[PASS] Fold 2: encoded dimension=41; train=3462; validation=866
[PASS] E02 Fold 3 | LR  | Macro-F1=0.5914 | Accuracy=0.6913 | RecallHigh=0.2545 | fit=0.027s
[PASS] E02 Fold 3 | SVM | Macro-F1=0.5479 | Accuracy=0.6821 | RecallHigh=0.1455 | fit=0.252s
[PASS] E02 Fold 3 | MLP | Macro-F1=0.4736 | Accuracy=0.6613 | RecallHigh=0.0182 | fit=0.187s
[PASS] Fold 3: encoded dimension=42

## Cell 6 — Kiểm tra consistency và lưu 5 file E02

Chỉ chạy sau khi đủ 5 folds. Tính metrics lại từ per-row predictions, kiểm tra mỗi model có đúng 4.328 validation predictions (mỗi sample đúng một lần), không có mẫu Final Test, rồi lưu dữ liệu full-precision và config. **Không ghi đè file đã tồn tại**.

In [6]:
assert set(fold_outputs) == set(range(5))
assert not any(path.exists() for path in OUTPUTS.values()), "Existing E02 results: refuse overwrite"

metrics_rows = [row for fid in sorted(fold_outputs) for row in fold_outputs[fid][0]]
prediction_rows = [row for fid in sorted(fold_outputs) for row in fold_outputs[fid][1]]
confusion_rows = [row for fid in sorted(fold_outputs) for row in fold_outputs[fid][2]]
metrics = pd.DataFrame(metrics_rows)
preds = pd.DataFrame(prediction_rows)
cm_rows = pd.DataFrame(confusion_rows)
assert len(metrics) == 15 and len(preds) == 4328 * 3
assert len(cm_rows) == 5 * 3 * 9
assert set(metrics["model"]) == set(MODELS)
label_by_id = dict(zip(labels["raw_row_index"].astype(int), y_array))

for name in MODELS:
    mp = preds.loc[preds["model"] == name].copy()
    assert len(mp) == 4328 and mp["raw_row_index"].is_unique
    assert set(mp["raw_row_index"].astype(int)) == dev_ids
    assert set(mp["raw_row_index"].astype(int)).isdisjoint(test_ids)
    assert all(label_by_id[int(rid)] == actual for rid, actual in
               zip(mp["raw_row_index"], mp["actual"]))

    agg_cm = confusion_matrix(mp["actual"], mp["predicted"], labels=CLASSES)
    for i, actual_cls in enumerate(CLASSES):
        for j, predicted_cls in enumerate(CLASSES):
            confusion_rows.append(dict(
                experiment="E02", scope="aggregate", fold="all",
                model=name, actual=actual_cls,
                predicted=predicted_cls, count=int(agg_cm[i,j])
            ))

    for fid in range(5):
        row = metrics.loc[(metrics["model"] == name) & (metrics["fold"] == fid)]
        mpf = mp.loc[mp["fold"] == fid]
        assert len(row) == 1 and len(mpf) in (865, 866)
        assert np.isclose(row.iloc[0]["macro_f1"], f1_score(
            mpf["actual"], mpf["predicted"], labels=CLASSES,
            average="macro", zero_division=0))
        assert np.isclose(row.iloc[0]["accuracy"], accuracy_score(
            mpf["actual"], mpf["predicted"]))

summary = (metrics.groupby("model", sort=False)
    .agg(folds=("fold", "count"),
         macro_f1_mean=("macro_f1", "mean"),
         macro_f1_sd=("macro_f1", "std"),
         accuracy_mean=("accuracy", "mean"),
         accuracy_sd=("accuracy", "std"),
         balanced_accuracy_mean=("balanced_accuracy", "mean"),
         recall_high_mean=("recall_high", "mean"),
         fit_seconds_mean=("fit_seconds", "mean"),
         predict_seconds_mean=("predict_seconds", "mean"))
    .reset_index().sort_values("macro_f1_mean", ascending=False))
assert len(summary) == 3 and summary["folds"].eq(5).all()

config = {
    "experiment_id": "E02",
    "experiment": "Strong_Classical",
    "dataset": "IWPC stable-dose eligible",
    "training_pool": "full_development_4328",
    "split_file": "data/splits/development_folds.csv",
    "cv": "locked_stratified_5fold",
    "models": list(MODELS),
    "feature_count_raw": len(approved),
    "features": approved,
    "representation": "Full_Features_After_Encoding_No_PCA",
    "preprocessing": "src.preprocessing.build_preprocessor fitted per train fold",
    "pca": None,
    "quantum_angle_scaler": None,
    "model_config": {
        "LR": {"max_iter": 2000, "random_state": 42},
        "SVM": {"kernel": "rbf", "C": 1.0, "gamma": "scale"},
        "MLP": {"hidden_layer_sizes": [32, 16], "max_iter": 500,
                "early_stopping": True, "n_iter_no_change": 20, "random_state": 42},
    },
    "class_weight": None,
    "class_order": CLASSES,
    "final_test_used_for_model_fit_or_evaluation": False,
    "retrospective_feature_policy_temporal_evidence": "UNVERIFIED",
    "environment": {"python": sys.version, "platform": platform.platform()},
    "versions": {pkg: version(pkg) for pkg in
                 ("numpy", "pandas", "scikit-learn", "PyYAML")},
    "checksums": {
        "raw_workbook": sha256_file(ROOT / "data/raw/PS206767-553247439.xls"),
        "split_manifest": sha256_file(manifest_path),
        "development_folds": sha256_file(SPLIT_DIR / "development_folds.csv"),
        "feature_policy": sha256_file(ROOT / "configs/feature_policy.yaml"),
        "experiment_matrix": sha256_file(ROOT / "docs/experiment_matrix.csv"),
    }
}

OUT.mkdir(parents=True, exist_ok=True)
for name, frame in (
    ("metrics", metrics),
    ("predictions", preds),
    ("confusion", pd.DataFrame(confusion_rows)),
    ("summary", summary),
):
    with OUTPUTS[name].open("x", encoding="utf-8", newline="") as f:
        frame.to_csv(f, index=False)
with OUTPUTS["config"].open("x", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2, allow_nan=False)

print("[PASS] E02: 15 model-fold runs; 12984 predictions; 4328 unique Development IDs per model")
print("\n=== STRONG CLASSICAL E02 — 5-FOLD SUMMARY ===")
display(summary.round(5))
print("\n[PASS] Saved 5 E02 artifacts to docs/results/")
for name, path in OUTPUTS.items():
    print(name, "=>", path)

[PASS] E02: 15 model-fold runs; 12984 predictions; 4328 unique Development IDs per model

=== STRONG CLASSICAL E02 — 5-FOLD SUMMARY ===


,model,folds,macro_f1_mean,macro_f1_sd,accuracy_mean,accuracy_sd,balanced_accuracy_mean,recall_high_mean,fit_seconds_mean,predict_seconds_mean
0,LR,5,0.58594,0.00772,0.68184,0.00795,0.56296,0.25636,0.02073,0.00035
2,MLP,5,0.54045,0.03929,0.66751,0.00388,0.52983,0.15818,0.28205,0.00085
1,SVM,5,0.53037,0.01649,0.66844,0.01259,0.51904,0.12909,0.26011,0.12755



[PASS] Saved 5 E02 artifacts to docs/results/
metrics => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\strong_classical_e02_metrics.csv
predictions => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\strong_classical_e02_predictions.csv
confusion => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\strong_classical_e02_confusion.csv
summary => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\strong_classical_e02_summary.csv
config => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\strong_classical_e02_config.json


## Diễn giải kết quả E02

- E02 là **strong reference riêng**: 4.328 mẫu Development + full encoded features; **không so trực tiếp như matched-input với E01 (600 mẫu, PCA k=4)**.
- So sánh macro-F1 (mean ± sample SD), accuracy, balanced accuracy, recall của từng lớp và confusion matrices. Chú ý nhóm High khi mô hình dự đoán ít hoặc không dự đoán High.
- `early_stopping=True` của MLP dùng validation nội bộ lấy từ **training fold**, không được nhầm với validation fold ngoài.
- `Final Test` vẫn khóa cho thí nghiệm cuối cùng; notebook này không sử dụng Test để fit/tune/eval.
- **Giữ notebook** này để tái lập E02. Không cần tạo script mới. Sau khi E02 hoàn tất, mới tổng hợp/chốt Main Benchmark k=4; **chưa bàn giao**.